# GPA Advisor Agent

## CSE476 - Agentic AI Project

### Project Objective

The GPA Advisor Agent is a tool-using AI agent designed to help students:

- Store course grades and credit values.
- Calculate their current credit-weighted GPA.
- Maintain grades across multiple conversation turns.
- Calculate the required future academic performance to reach a target GPA.
- Detect mathematically impossible GPA targets.

The project follows an agentic architecture based on:

**THINK → ACT → OBSERVE → REASON → NEXT ACTION**

The Large Language Model decides which tool should be used, while deterministic Python tools perform GPA calculations.

# Agent Architecture

The GPA Advisor follows the following workflow:

User Request
    ↓
Large Language Model
    ↓
Decides which tool is required
    ↓
Tool Execution
    ↓
Observation / Tool Result
    ↓
Large Language Model
    ↓
Next Action or Final Answer

The available tools are:

1. `add_grade()`
2. `compute_gpa()`
3. `calculate_target_gpa()`

The agent maintains session memory using the `GPAMemory` class.

In [1]:
from gpa_agent import (
    run_agent,
    clear_session,
    show_memory,
)

# Demonstration 1: Multi-Step Agent Behaviour

In this demonstration, the agent must:

1. Store three course grades.
2. Calculate the current GPA.
3. Calculate the required average grade point to reach a target GPA.

The agent decides which tools to call and uses the results of previous tool calls to determine the next action.

In [2]:
clear_session()

answer, history = run_agent(
    """
    Add Artificial Intelligence with grade A and 4 credits.
    Add Data Mining with grade B+ and 3 credits.
    Add Cloud Computing with grade A+ and 3 credits.

    Calculate my current GPA.

    I have 15 credits remaining.
    What average grade point do I need to reach
    a final GPA of 8.5?
    """,
    verbose=True,
)

print("\nFINAL ANSWER:\n")
print(answer)

print("\nSESSION MEMORY:\n")
print(show_memory())


[step 1] TOOL: add_grade({'course': 'Artificial Intelligence', 'credits': 4, 'grade': 'A'})
[step 1] RESULT:
Added Artificial Intelligence with grade A and 4.0 credits.

[step 2] TOOL: add_grade({'course': 'Data Mining', 'credits': 3, 'grade': 'B+'})
[step 2] RESULT:
Added Data Mining with grade B+ and 3.0 credits.

[step 3] TOOL: add_grade({'course': 'Cloud Computing', 'credits': 3, 'grade': 'A+'})
[step 3] RESULT:
Added Cloud Computing with grade A+ and 3.0 credits.

[step 4] TOOL: compute_gpa({})
[step 4] RESULT:
Current GPA: 8.00
Courses counted: 3
Total credits: 10.0

[step 5] TOOL: calculate_target_gpa({'remaining_credits': 15, 'target_gpa': 8.5})
[step 5] RESULT:
Target GPA: 8.50
Current GPA: 8.00
Completed credits: 10.0
Remaining credits: 15.0
Required average grade point in remaining courses: 8.83/10
Status: Mathematically ACHIEVABLE because the required average (8.83) is <= the maximum possible grade point of 10.0 (Grade O).
Advice: The student can reach this target by earni

# Demonstration 2: Memory Across Separate Turns

A key property of an agent is the ability to maintain useful state.

In this demonstration, grades are added across separate conversation turns.

The final GPA calculation should use information stored during previous turns.

In [3]:
clear_session()

history = None


print("TURN 1\n")

answer, history = run_agent(
    "Add Machine Learning with grade A and 4 credits.",
    conversation_history=history,
    verbose=True,
)

print("\nAgent Answer:")
print(answer)

TURN 1


[step 1] TOOL: add_grade({'course': 'Machine Learning', 'credits': 4, 'grade': 'A'})
[step 1] RESULT:
Added Machine Learning with grade A and 4.0 credits.
[step 2] DONE

Agent Answer:
✅ Added **Machine Learning** (4 credits) with grade **A**. Let me know if you’d like to add more courses, check your current GPA, or see your course history!


In [4]:
print("TURN 2\n")

answer, history = run_agent(
    "Add Deep Learning with grade A+ and 3 credits.",
    conversation_history=history,
    verbose=True,
)

print("\nAgent Answer:")
print(answer)

TURN 2


[step 1] TOOL: add_grade({'course': 'Deep Learning', 'credits': 3, 'grade': 'A+'})
[step 1] RESULT:
Added Deep Learning with grade A+ and 3.0 credits.
[step 2] DONE

Agent Answer:
✅ Added **Deep Learning** (3 credits) with grade **A+**. Anything else?


In [5]:
print("TURN 3\n")

answer, history = run_agent(
    "What is my current GPA?",
    conversation_history=history,
    verbose=True,
)

print("\nAgent Answer:")
print(answer)

print("\nMEMORY:")
print(show_memory())

TURN 3


[step 1] TOOL: compute_gpa({})
[step 1] RESULT:
Current GPA: 8.43
Courses counted: 2
Total credits: 7.0
[step 2] DONE

Agent Answer:
Your current GPA is **8.43** based on 7 total credits (Machine Learning – A, Deep Learning – A+). Let me know if you’d like to add more courses, remove one, or check your transcript!

MEMORY:
Machine Learning: Grade A, Credits 4.0
Deep Learning: Grade A+, Credits 3.0


# Demonstration 3: Impossible Target GPA

The agent should handle edge cases instead of always providing an unrealistic recommendation.

In this test:

- The current GPA is relatively low.
- Only one credit remains.
- The student requests a final GPA of 10.

The GPA tool should determine whether the required future performance is mathematically possible.

In [6]:
clear_session()

answer, history = run_agent(
    """
    Add Database Systems with grade B and 10 credits.

    I have only 1 credit remaining.

    What average grade point do I need to reach
    a final GPA of 10?
    """,
    verbose=True,
)

print("\nFINAL ANSWER:\n")
print(answer)


[step 1] TOOL: add_grade({'course': 'Database Systems', 'credits': 10, 'grade': 'B'})
[step 1] RESULT:
Added Database Systems with grade B and 10.0 credits.

[step 2] TOOL: calculate_target_gpa({'remaining_credits': 1, 'target_gpa': 10})
[step 2] RESULT:
Target GPA: 10.00
Current GPA: 6.00
Remaining credits: 1.0

Required future average: 50.00/10
Status: Mathematically IMPOSSIBLE because the required average (50.00) exceeds the maximum possible grade point of 10.0 (Grade O).
[step 3] DONE

FINAL ANSWER:

It’s impossible to reach a final GPA of **10.00** with only 1 credit left.  
Your current GPA is 6.00 and, even if you earned a perfect “O” (10.0) in that last credit, the weighted average would still fall short. The calculation shows a required future average of **50.00**, which is far above the maximum 10.0 grade point.  

If you have any other questions or need help planning a different target GPA, let me know!


# Conclusion

The GPA Advisor successfully demonstrates an agentic AI workflow.

The system:

- Uses a Large Language Model for decision-making.
- Uses deterministic Python tools for GPA calculations.
- Maintains course information in session memory.
- Performs multiple sequential tool calls.
- Uses tool observations to determine subsequent actions.
- Supports multi-turn interactions.
- Handles mathematically impossible target GPA requests.

The project therefore demonstrates the difference between a simple LLM response and an AI agent capable of interacting with tools, maintaining state, and completing multi-step tasks.